# Speed-up stages on Colab L4 (plan: `worklog/plans/2026-11-20-speedup-plan.md`)
Open on colab.research.google.com; **Runtime -> Change runtime type -> L4 GPU**. Never paste the HF token into a cell.

- **tune** = `train-00000` (all tuning), **holdout** = `train-00024` + `train-00049` (scored only, never tuned on).
- References and logs live on Drive: `DRIVE=/content/drive/MyDrive/vidx` (`reference/`, `frames/`, `bench_log.csv`).
- Accept bar (tune AND holdout): F1 >= 0.985, people exact >= 0.93, within +-1 >= 0.985.
- Fallback ladder, fixed in advance: k=3 -> k=5 -> every frame fp16 -> every frame fp32.

**After every runtime reset:** run section 0 (all of it), then the step you were on. Long runs: wait for them to finish
(`tail` the log) before running anything that reads their output.

## 0. Setup (re-run after every reset)
### 0.1 Machine, venv, code, model

In [ ]:
%%bash
set -e
nvidia-smi --query-gpu=name,memory.total --format=csv; nproc; free -h | head -2; df -h /content | tail -1
cd /content
pip install -q uv
[ -d /content/venv ] || uv venv --python 3.11 /content/venv
[ -d video-indexing-yolo11 ] || git clone https://github.com/sobanali256/video-indexing-yolo11.git
cd video-indexing-yolo11 && git pull -q && git log --oneline -1
apt-get install -y -q libgl1 libglib2.0-0 > /dev/null
uv pip install -q --index-strategy unsafe-best-match --python /content/venv/bin/python -r requirements.txt
/content/venv/bin/python -c "import vidx,torch,av,ultralytics;print(torch.__version__,torch.cuda.is_available(),av.__version__,ultralytics.__version__)"
/content/venv/bin/python prepare.py

### 0.2 Hugging Face login (Colab Secret `HF_TOKEN`, else prompt)

In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata
    login(userdata.get("HF_TOKEN"))
except Exception:
    import getpass
    login(getpass.getpass("HF token: "))

### 0.3 Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!ls /content/drive/MyDrive/vidx /content/drive/MyDrive/vidx/reference

### 0.4a Download the 3 reference files first (foreground, a few minutes; one call per file, see WORKLOG issue #9)

In [ ]:
%%bash
for n in 00000 00024 00049; do
  /content/venv/bin/hf download HuggingFaceFV/finevideo --repo-type dataset --include "data/train-$n-*" --local-dir /content/finevideo > /dev/null 2>&1     && echo "train-$n: ok" || echo "train-$n: FAILED (HF login done? disk space?)"
done
ls -l /content/finevideo/data; df -h /content | tail -1

### 0.4b Start the 50-file download in the background (skips files already complete; `download.pid` tells whether it is still running)

In [ ]:
!if [ -f /content/download.pid ] && kill -0 "$(cat /content/download.pid)" 2>/dev/null; then echo "download already running (pid $(cat /content/download.pid))"; else nohup /content/venv/bin/hf download HuggingFaceFV/finevideo --repo-type dataset --include "data/train-000[0-4]*" --local-dir /content/finevideo > /content/download.log 2>&1 & echo $! > /content/download.pid; echo "download started (pid $(cat /content/download.pid))"; fi
!sleep 20; tail -c 300 /content/download.log; echo; ls /content/finevideo/data 2>/dev/null | wc -l

### 0.5 Download status + tune/holdout links (re-run until the 3 reference files are there)

In [ ]:
%%bash
tail -c 200 /content/download.log; echo
D=/content/finevideo/data
echo "parquet files: $(ls $D/*.parquet 2>/dev/null | wc -l) / 50, size $(du -sh /content/finevideo 2>/dev/null | cut -f1)"
mkdir -p /content/subset/tune /content/subset/holdout
for f in $D/train-00000-*.parquet; do [ -e "$f" ] && ln -sf "$f" /content/subset/tune/; done
for f in $D/train-00024-*.parquet $D/train-00049-*.parquet; do [ -e "$f" ] && ln -sf "$f" /content/subset/holdout/; done
ls -l /content/subset/tune /content/subset/holdout

## 1. Stage 1a: sampling simulator (CPU only, needs only Drive)
The `check k=0` line must be 1.0000. Results 2026-11-20: k=3 accepted (F1 0.9880, exact 0.9584, within1 0.9988).

In [ ]:
%%bash
cd /content/video-indexing-yolo11
D=/content/drive/MyDrive/vidx
echo "dumps: $(ls $D/frames | wc -l) (expect 33)"
/content/venv/bin/python tools/simulate_sampling.py --dumps $D/frames --ref $D/reference/ref_tune.jsonl   --k 1,2,3,4,5,6,7,9 --adaptive 3:7,3:9,5:9 --json $D/sim_tune.json

## 2. Stage 0: profile (needs the tune file). Done 2026-11-20
fp32 99 fps, fp16 155 fps (accuracy 0.9995 / 0.998 / 1.0); decode ~0.2 ms/frame. `fail_accuracy` here is expected (`--only-common` = PARTIAL).

In [ ]:
%%bash
cd /content/video-indexing-yolo11
D=/content/drive/MyDrive/vidx; PY=/content/venv/bin/python
$PY tools/profile_decode.py --input /content/subset/tune --videos 5
for P in fp32 fp16; do
  $PY bench.py run --label "profile $P every frame" --input /content/subset/tune --out /content/prof_$P.jsonl     --ref tune=$D/reference/ref_tune.jsonl --only-common --log $D/bench_log.csv -- --limit-videos 5 --sample 0 --decode-threads 0 --precision $P     2>&1 | grep -E "^(PASS|FAIL|PARTIAL|bench|videos)"
done
tail -2 $D/bench_log.csv

## 3. Stage 1 gate: k=3 + FP16
1. k=3 FP32 on tune must reproduce the simulator exactly (F1 0.9880, exact 0.9584, within1 0.9988): proves the decoder samples the simulated frames.
2. k=3 FP16 on tune: accept bar.
3. k=3 FP16 on holdout: accept bar (holdout is never used to choose anything; if it fails, step down the ladder).

In [ ]:
%%bash
# setup guard: stop early (instead of a pile of errors) if section 0 has not run in this runtime
for p in /content/video-indexing-yolo11 /content/venv/bin/python /content/drive/MyDrive/vidx/reference/ref_tune.jsonl; do
  [ -e "$p" ] || { echo "MISSING $p -> the runtime was reset: run section 0 first"; exit 1; }
done
for d in tune holdout; do
  [ -n "$(ls /content/subset/$d/*.parquet 2>/dev/null)" ] || { echo "MISSING /content/subset/$d/*.parquet -> run 0.4a and 0.5 first"; exit 1; }
done
nvidia-smi --query-gpu=name --format=csv,noheader
cd /content/video-indexing-yolo11 && git pull -q && git log --oneline -1
D=/content/drive/MyDrive/vidx; PY=/content/venv/bin/python; R=$D/reference
run() {  # label input out ref -- run.py args
  $PY bench.py run --label "$1" --input "$2" --out "$3" --ref "$4" --log $D/bench_log.csv -- "${@:5}"     2>&1 | grep -E "^(PASS|FAIL|bench|videos|overall)"
  cp "$3" $D/
}
run "stage1 k3 fp32 tune"    /content/subset/tune    /content/s1_k3_fp32_tune.jsonl    tune=$R/ref_tune.jsonl       --sample 3 --precision fp32
run "stage1 k3 fp16 tune"    /content/subset/tune    /content/s1_k3_fp16_tune.jsonl    tune=$R/ref_tune.jsonl       --sample 3 --precision fp16
run "stage1 k3 fp16 holdout" /content/subset/holdout /content/s1_k3_fp16_holdout.jsonl holdout=$R/ref_holdout.jsonl --sample 3 --precision fp16
tail -3 $D/bench_log.csv

## 4. Stage 2a: decode threads + cross-video batches + direct inference (`--decode-threads N`)
Defaults are k=3 + FP16 (Stage 1). Stage 2a only changes HOW frames reach the GPU, not which frames.
1. **4.1 direct path vs predict()** on the same 1000 frames and batches: FP32 must be 1.0000, FP16 >= 0.999.
2. **4.2 timing on tune** with 1 and 2 decode threads (accept bar must still hold).
3. **4.3 holdout** with the faster thread count. If 4.1-4.3 pass, the default becomes that thread count.

Use an **L4** runtime for the timings (Stage 1 numbers above were on a T4).

### 4.1 Direct path vs predict() (~5 min)

In [ ]:
%%bash
# setup guard: stop early (instead of a pile of errors) if section 0 has not run in this runtime
for p in /content/video-indexing-yolo11 /content/venv/bin/python /content/drive/MyDrive/vidx/reference/ref_tune.jsonl; do
  [ -e "$p" ] || { echo "MISSING $p -> the runtime was reset: run section 0 first"; exit 1; }
done
for d in tune holdout; do
  [ -n "$(ls /content/subset/$d/*.parquet 2>/dev/null)" ] || { echo "MISSING /content/subset/$d/*.parquet -> run 0.4a and 0.5 first"; exit 1; }
done
nvidia-smi --query-gpu=name --format=csv,noheader
cd /content/video-indexing-yolo11 && git pull -q && git log --oneline -1
PY=/content/venv/bin/python
for P in fp32 fp16; do
  $PY tools/batch_agreement.py --input /content/subset/tune --frames 1000 --direct --precision $P 2>&1 | grep -E "frames collected|direct"
done

### 4.2 Timing on tune: Stage 1 path (threads 0) vs 1 and 2 decode threads (~6-8 min each on an L4)

In [ ]:
%%bash
# setup guard: stop early (instead of a pile of errors) if section 0 has not run in this runtime
for p in /content/video-indexing-yolo11 /content/venv/bin/python /content/drive/MyDrive/vidx/reference/ref_tune.jsonl; do
  [ -e "$p" ] || { echo "MISSING $p -> the runtime was reset: run section 0 first"; exit 1; }
done
for d in tune holdout; do
  [ -n "$(ls /content/subset/$d/*.parquet 2>/dev/null)" ] || { echo "MISSING /content/subset/$d/*.parquet -> run 0.4a and 0.5 first"; exit 1; }
done
nvidia-smi --query-gpu=name --format=csv,noheader
cd /content/video-indexing-yolo11 && git pull -q
D=/content/drive/MyDrive/vidx; PY=/content/venv/bin/python; R=$D/reference
GPU=$(nvidia-smi --query-gpu=name --format=csv,noheader | tr ' ' '_')
for T in 0 1 2; do
  $PY bench.py run --label "stage2a threads$T tune $GPU" --input /content/subset/tune --out /content/s2_t${T}_tune.jsonl \
    --ref tune=$R/ref_tune.jsonl --log $D/bench_log.csv -- --decode-threads $T 2>&1 | grep -E "^(PASS|FAIL|bench|videos|overall)"
  cp /content/s2_t${T}_tune.jsonl $D/
done
tail -3 $D/bench_log.csv

### 4.3 Holdout with the faster thread count (set `T` from 4.2)

In [ ]:
%%bash
# setup guard: stop early (instead of a pile of errors) if section 0 has not run in this runtime
for p in /content/video-indexing-yolo11 /content/venv/bin/python /content/drive/MyDrive/vidx/reference/ref_tune.jsonl; do
  [ -e "$p" ] || { echo "MISSING $p -> the runtime was reset: run section 0 first"; exit 1; }
done
for d in tune holdout; do
  [ -n "$(ls /content/subset/$d/*.parquet 2>/dev/null)" ] || { echo "MISSING /content/subset/$d/*.parquet -> run 0.4a and 0.5 first"; exit 1; }
done
nvidia-smi --query-gpu=name --format=csv,noheader
T=2   # <- the faster thread count from 4.2
cd /content/video-indexing-yolo11
D=/content/drive/MyDrive/vidx; PY=/content/venv/bin/python; R=$D/reference
GPU=$(nvidia-smi --query-gpu=name --format=csv,noheader | tr ' ' '_')
$PY bench.py run --label "stage2a threads$T holdout $GPU" --input /content/subset/holdout --out /content/s2_t${T}_holdout.jsonl \
  --ref holdout=$R/ref_holdout.jsonl --log $D/bench_log.csv -- --decode-threads $T 2>&1 | grep -E "^(PASS|FAIL|bench|videos|overall)"
cp /content/s2_t${T}_holdout.jsonl $D/
tail -1 $D/bench_log.csv

## 5. Full 50-file run -> results.jsonl.gz (default settings only)
The run uses **only default flags** (plus checkpoint/stats flags, which do not change results), so the tagged commit's
`run.py` defaults are exactly the configuration that produced `results.jsonl.gz`.
Expected on the L4: ~12.98 M frames at ~1,900 fps -> **about 2 h**. Checkpoints go to Drive every 10 min.
**After a runtime reset:** section 0 (0.1-0.4b, wait for the download), then **5.4** (resume).

### 5.1 Download complete? (must show 50 files and no running download)

In [ ]:
%%bash
set -e
D=/content/drive/MyDrive/vidx; PY=/content/venv/bin/python; DATA=/content/finevideo/data
for p in /content/video-indexing-yolo11 $PY $D; do [ -e "$p" ] || { echo "MISSING $p -> run section 0 first"; exit 1; }; done
cd /content/video-indexing-yolo11
n=$(ls $DATA/*.parquet 2>/dev/null | wc -l)
running=no; [ -f /content/download.pid ] && kill -0 "$(cat /content/download.pid)" 2>/dev/null && running=yes
echo "parquet files: $n / 50, download running: $running, size $(du -sh /content/finevideo | cut -f1)"
tail -c 200 /content/download.log 2>/dev/null; echo
[ "$n" = 50 ] && [ "$running" = no ] && echo "READY for 5.2" || echo "NOT READY: wait, then re-run this cell" 

### 5.2 Launch the full run in the background (once)

In [ ]:
%%bash
set -e
D=/content/drive/MyDrive/vidx; PY=/content/venv/bin/python; DATA=/content/finevideo/data
for p in /content/video-indexing-yolo11 $PY $D; do [ -e "$p" ] || { echo "MISSING $p -> run section 0 first"; exit 1; }; done
cd /content/video-indexing-yolo11
n=$(ls $DATA/*.parquet | wc -l); [ "$n" = 50 ] || { echo "only $n/50 files: wait for 5.1"; exit 1; }
git fetch -q --tags origin; git checkout -q results-v1  # the tagged configuration, never a newer main
git log --oneline -1 | tee $D/full50.commit.txt; git describe --tags --exact-match 2>/dev/null | tee -a $D/full50.commit.txt || echo "(commit not tagged)"
nvidia-smi --query-gpu=name --format=csv,noheader | tee -a $D/full50.commit.txt
nohup $PY run.py --input $DATA --output /content/results.jsonl \
  --checkpoint-dir $D/ckpt --run-name full50 --stats-json $D/full50.stats.json > /content/full50.log 2>&1 &
echo $! > /content/full50.pid; sleep 60; tail -n 5 /content/full50.log

### 5.3 Progress (re-run any time)

In [ ]:
!tail -n 3 /content/full50.log; echo; [ -f /content/full50.pid ] && kill -0 $(cat /content/full50.pid) 2>/dev/null && echo "RUNNING" || echo "NOT RUNNING (finished, crashed or reset: check the log)"; wc -l < /content/results.jsonl.done 2>/dev/null; nvidia-smi --query-gpu=utilization.gpu --format=csv,noheader

### 5.4 Resume after a reset (same command + `--resume`; restores the newest Drive checkpoint)

In [ ]:
%%bash
set -e
D=/content/drive/MyDrive/vidx; PY=/content/venv/bin/python; DATA=/content/finevideo/data
for p in /content/video-indexing-yolo11 $PY $D; do [ -e "$p" ] || { echo "MISSING $p -> run section 0 first"; exit 1; }; done
cd /content/video-indexing-yolo11
n=$(ls $DATA/*.parquet | wc -l); [ "$n" = 50 ] || { echo "only $n/50 files: wait for 5.1"; exit 1; }
# resume with exactly the code that started the run (recorded by 5.2), never a newer main
C=$(head -1 $D/full50.commit.txt | cut -d' ' -f1); [ -n "$C" ] || { echo "no $D/full50.commit.txt: was 5.2 run?"; exit 1; }
git fetch -q --tags origin; git checkout -q "$C"; echo "resuming with $(git log --oneline -1)"
nohup $PY run.py --input $DATA --output /content/results.jsonl --resume \
  --checkpoint-dir $D/ckpt --run-name full50 --stats-json $D/full50.stats.json >> /content/full50.log 2>&1 &
echo $! > /content/full50.pid; sleep 60; tail -n 5 /content/full50.log

### 5.5 Finish: schema, sanity scores on the 3 reference files, gzip to Drive, total time

In [ ]:
%%bash
set -e
D=/content/drive/MyDrive/vidx; PY=/content/venv/bin/python; DATA=/content/finevideo/data
for p in /content/video-indexing-yolo11 $PY $D; do [ -e "$p" ] || { echo "MISSING $p -> run section 0 first"; exit 1; }; done
cd /content/video-indexing-yolo11
[ -f /content/full50.pid ] && kill -0 "$(cat /content/full50.pid)" 2>/dev/null && { echo "still running: wait"; exit 1; }
tail -n 2 /content/full50.log
$PY tools/check_schema.py /content/results.jsonl
$PY score.py --ref tune=$D/reference/ref_tune.jsonl --ref holdout=$D/reference/ref_holdout.jsonl /content/results.jsonl | tail -4
gzip -9 -c /content/results.jsonl > $D/results.jsonl.gz && ls -l $D/results.jsonl.gz
cat $D/full50.stats.json
G=$(ls -d $D/ckpt/full50/gen-* | tail -1); cat $G/progress.json

## 6. Final checks (grader readiness)
Run on an **L4** after section 0. Each cell prints PASS/FAIL or the facts to paste back.
- 6.1 OS + disk type (for HARDWARE.md)
- 6.2 decode threads: identical output for 2 / 3 / 4 threads + speed (bench_log.csv). Performance-only change after results-v1.
- 6.3 rotation: scan all 50 files for rotation tags + compare our decoder with ffmpeg on rotated copies
- 6.4 damaged inputs (truncated / random / null / empty mp4, corrupt and truncated parquet)
- 6.5 offline run (network blocked)
- 6.6 clean `pip install -r requirements.txt` in a fresh venv + `pip freeze` (exact pins of every dependency)

### 6.1 OS and disk

In [ ]:
!cat /etc/os-release | head -4; uname -r; lsblk -d -o NAME,ROTA,SIZE,MODEL; df -h /content | tail -1

### 6.2 Decode threads: same output for 2 / 3 / 4 threads, and speed (~2.5 min per run on the L4)
`auto` = one thread per 4 CPUs, 2..4 (3 on Colab's 12 vCPUs, 4 on the 32-thread grader). Every run must give
**byte-identical** output after sorting (`identical: yes`).

In [ ]:
%%bash
# setup guard: stop early if section 0 has not run in this runtime
for p in /content/video-indexing-yolo11 /content/venv/bin/python /content/drive/MyDrive/vidx/reference/ref_tune.jsonl; do
  [ -e "$p" ] || { echo "MISSING $p -> the runtime was reset: run section 0 first"; exit 1; }
done
[ -n "$(ls /content/subset/tune/*.parquet 2>/dev/null)" ] || { echo "MISSING tune parquet -> run 0.4a and 0.5 first"; exit 1; }
cd /content/video-indexing-yolo11 && git checkout -q main && git pull -q && git log --oneline -1
D=/content/drive/MyDrive/vidx; PY=/content/venv/bin/python; R=$D/reference
GPU=$(nvidia-smi --query-gpu=name --format=csv,noheader | tr ' ' '_')
echo "auto resolves to $($PY -c 'import run; print(run.auto_decode_threads())') threads on $(nproc) CPUs"
for T in 2 3 4; do
  $PY bench.py run --label "threads$T tune perf $GPU" --input /content/subset/tune --out /content/t${T}_tune.jsonl --ref tune=$R/ref_tune.jsonl --log $D/bench_log.csv -- --decode-threads $T 2>&1 | grep -E "^(PASS|FAIL|bench|videos)"
  cp /content/t${T}_tune.jsonl $D/
done
for T in 3 4; do
  n=$(diff <(sort /content/t2_tune.jsonl) <(sort /content/t${T}_tune.jsonl) | grep -c '^[<>]')
  if [ "$n" = 0 ]; then echo "threads 2 vs $T: identical: yes ($(wc -l < /content/t2_tune.jsonl) lines)"; else echo "threads 2 vs $T: identical: NO ($n differing lines)"; $PY score.py /content/t2_tune.jsonl /content/t${T}_tune.jsonl | tail -3; fi
done
tail -3 $D/bench_log.csv | cut -d, -f1,2,4,5,10,11,14,15

### 6.3 Rotation: tags in the 50 files, and our decoder vs ffmpeg on rotated copies

In [ ]:
%%bash
# setup guard: stop early if section 0 has not run in this runtime
for p in /content/video-indexing-yolo11 /content/venv/bin/python /content/drive/MyDrive/vidx/reference/ref_tune.jsonl; do
  [ -e "$p" ] || { echo "MISSING $p -> the runtime was reset: run section 0 first"; exit 1; }
done
[ -n "$(ls /content/subset/tune/*.parquet 2>/dev/null)" ] || { echo "MISSING tune parquet -> run 0.4a and 0.5 first"; exit 1; }
cd /content/video-indexing-yolo11 && git checkout -q main && git pull -q && git log --oneline -1
D=/content/drive/MyDrive/vidx; PY=/content/venv/bin/python; R=$D/reference
GPU=$(nvidia-smi --query-gpu=name --format=csv,noheader | tr ' ' '_')
which ffmpeg > /dev/null || apt-get install -y -q ffmpeg > /dev/null
ffmpeg -version | head -1
echo "== rotation tags in the 50 files =="
$PY tools/find_rotated.py --input /content/finevideo/data --max-files 50 2>&1 | tail -8
echo "== decoder vs ffmpeg (ground truth) =="
$PY tools/rotation_check.py --input /content/subset/tune --row 0 --work /content/rotcheck

### 6.4 Damaged inputs (must PASS; the run must exit 0)

In [ ]:
%%bash
# setup guard: stop early if section 0 has not run in this runtime
for p in /content/video-indexing-yolo11 /content/venv/bin/python /content/drive/MyDrive/vidx/reference/ref_tune.jsonl; do
  [ -e "$p" ] || { echo "MISSING $p -> the runtime was reset: run section 0 first"; exit 1; }
done
[ -n "$(ls /content/subset/tune/*.parquet 2>/dev/null)" ] || { echo "MISSING tune parquet -> run 0.4a and 0.5 first"; exit 1; }
cd /content/video-indexing-yolo11 && git checkout -q main && git pull -q && git log --oneline -1
D=/content/drive/MyDrive/vidx; PY=/content/venv/bin/python; R=$D/reference
GPU=$(nvidia-smi --query-gpu=name --format=csv,noheader | tr ' ' '_')
rm -rf /content/damaged /content/damaged.jsonl*
$PY tools/make_damaged.py make --input /content/subset/tune --out /content/damaged
$PY run.py --input /content/damaged --output /content/damaged.jsonl 2> /content/damaged.log; echo "run.py exit code: $?"
grep -iE "warning|error" /content/damaged.log | tail -5
$PY tools/check_schema.py /content/damaged.jsonl
$PY tools/make_damaged.py check --out /content/damaged --results /content/damaged.jsonl

### 6.5 Offline run: every socket connection is blocked (and the network namespace too, if allowed)
Must show `blocked connection attempts: 0`, exit code 0, and the same lines as the online run for those videos.

In [ ]:
%%bash
# setup guard: stop early if section 0 has not run in this runtime
for p in /content/video-indexing-yolo11 /content/venv/bin/python /content/drive/MyDrive/vidx/reference/ref_tune.jsonl; do
  [ -e "$p" ] || { echo "MISSING $p -> the runtime was reset: run section 0 first"; exit 1; }
done
[ -n "$(ls /content/subset/tune/*.parquet 2>/dev/null)" ] || { echo "MISSING tune parquet -> run 0.4a and 0.5 first"; exit 1; }
cd /content/video-indexing-yolo11 && git checkout -q main && git pull -q && git log --oneline -1
D=/content/drive/MyDrive/vidx; PY=/content/venv/bin/python; R=$D/reference
GPU=$(nvidia-smi --query-gpu=name --format=csv,noheader | tr ' ' '_')
mkdir -p /content/netblock
cat > /content/netblock/sitecustomize.py <<'PY'
import socket, sys
def _blocked(*a, **k):
    sys.stderr.write(f"NETWORK BLOCKED: {a[1:2] or a}\n")
    raise OSError("network disabled for the offline test")
socket.socket.connect = _blocked
socket.socket.connect_ex = _blocked
socket.create_connection = _blocked
PY
PRE=""; unshare -n true 2>/dev/null && PRE="unshare -n" && echo "network namespace: isolated (unshare -n)"
rm -f /content/offline.jsonl*
PYTHONPATH=/content/netblock $PRE $PY run.py --input /content/subset/tune --output /content/offline.jsonl --limit-videos 3 2> /content/offline.log; echo "run.py exit code: $?"
echo "blocked connection attempts: $(grep -c 'NETWORK BLOCKED' /content/offline.log)"
grep 'NETWORK BLOCKED' /content/offline.log | sort | uniq -c | head
python3 - <<'PY'
import json
off = sorted(open("/content/offline.jsonl").read().splitlines())
vids = {json.loads(l)["video"] for l in off}
ref = sorted(l for l in open("/content/t2_tune.jsonl").read().splitlines() if json.loads(l)["video"] in vids)
print(f"offline videos {len(vids)}, lines {len(off)}; same as the online run: {'yes' if off == ref else 'NO'}")
PY

### 6.6 Clean install exactly as the README says (plain pip, fresh Python 3.11 venv) + freeze (~5-10 min)
Paste the output; the freeze file on Drive becomes the exact pins for every dependency.

In [ ]:
%%bash
# setup guard: stop early if section 0 has not run in this runtime
for p in /content/video-indexing-yolo11 /content/venv/bin/python /content/drive/MyDrive/vidx/reference/ref_tune.jsonl; do
  [ -e "$p" ] || { echo "MISSING $p -> the runtime was reset: run section 0 first"; exit 1; }
done
[ -n "$(ls /content/subset/tune/*.parquet 2>/dev/null)" ] || { echo "MISSING tune parquet -> run 0.4a and 0.5 first"; exit 1; }
cd /content/video-indexing-yolo11 && git checkout -q main && git pull -q && git log --oneline -1
D=/content/drive/MyDrive/vidx; PY=/content/venv/bin/python; R=$D/reference
GPU=$(nvidia-smi --query-gpu=name --format=csv,noheader | tr ' ' '_')
rm -rf /content/venv_pip
uv venv -q --seed --python 3.11 /content/venv_pip
/content/venv_pip/bin/python -m pip install -q -r requirements.txt 2>&1 | tail -5; echo "pip exit code: ${PIPESTATUS[0]}"
/content/venv_pip/bin/python -m pip check
/content/venv_pip/bin/python -m pip freeze > $D/freeze_py311.txt; wc -l $D/freeze_py311.txt
/content/venv_pip/bin/python -c "import vidx, torch, av, cv2, pyarrow, ultralytics; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'ultralytics', ultralytics.__version__, 'av', av.__version__)"
/content/venv_pip/bin/python prepare.py 2>&1 | tail -1
rm -f /content/pip_check.jsonl*
/content/venv_pip/bin/python run.py --input /content/subset/tune --output /content/pip_check.jsonl --limit-videos 2 2>&1 | tail -1
cat $D/freeze_py311.txt

## Helpers

In [ ]:
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv; pgrep -af "[r]un.py|[h]f download" | cut -c1-150; tail -c 300 /content/download.log